In [1]:
from pathlib import Path

project_dir = Path(r"D:\Projects\NewsGPT")
app_js = project_dir / "frontend" / "app.js"
html_file = project_dir / "frontend" / "index.html"

js = app_js.read_text(encoding="utf-8")
html = html_file.read_text(encoding="utf-8")

for i, line in enumerate(js.splitlines(), start=1):
    if any(term in line.lower() for term in [
        "async function viewsearch",
        "function viewsearch",
        "function render",
        "function navigate",
        "function router",
        "function viewhome",
        "function initsearch",
    ]):
        print(f"app.js:{i}: {line.strip()}")

print("\n--- HTML layout markers ---")

for i, line in enumerate(html.splitlines(), start=1):
    if any(term in line.lower() for term in [
        "search",
        "main",
        "content",
        "script",
    ]):
        print(f"index.html:{i}: {line.strip()}")

app.js:236: function initSearch(root) {
app.js:242: function render() {
app.js:400: function viewHome(scrollTo) {
app.js:448: async function viewSearch(q) {
app.js:538: function renderTL() {

--- HTML layout markers ---
index.html:5: <meta name="viewport" content="width=device-width, initial-scale=1">
index.html:7: <meta name="description" content="NewsGPT groups related coverage into story timelines and keeps the original sources.">
index.html:14: <a class="skip-link" href="#app">Skip to content</a>
index.html:95: <span class="masthead__inscription" aria-hidden="true">Anno MMXXVI</span>
index.html:111: <nav class="drawer" id="drawer" aria-label="Main navigation" aria-hidden="true" inert>
index.html:124: <!-- ============ MAIN ============ -->
index.html:125: <main id="app" tabindex="-1"></main>
index.html:172: <script src="data.js"></script>
index.html:173: <script src="app.js"></script>


In [2]:
from pathlib import Path

app_path = Path(r"D:\Projects\NewsGPT\frontend\app.js")
lines = app_path.read_text(encoding="utf-8").splitlines()

start = next(
    i for i, line in enumerate(lines)
    if "async function viewSearch(q)" in line
)

end = next(
    i for i, line in enumerate(lines[start + 1:], start + 1)
    if "story / timeline" in line
)

print("\n".join(
    f"{i + 1}: {lines[i]}"
    for i in range(start, end)
))

448:   async function viewSearch(q) {
449:     app.innerHTML = `<div class="container page-pad">
450:       <p class="crumbs"><a href="#/">← Back to home</a></p>
451:       <div class="section-head"><h2>Search</h2><span class="kicker" id="srCount">Searching…</span></div>
452:       <div style="max-width:560px;margin-bottom:26px">${searchMarkup(q)}</div>
453:       <div id="srOut" aria-live="polite"></div></div>`;
454:     initSearch($(".search", app));
455:     const out = $("#srOut");
456:     try {
457:       const r = await searchNews(q);
458:       $("#srCount").textContent = `${r.length} result${r.length === 1 ? "" : "s"} for “${q}”`;
459:       out.innerHTML = r.length
460:   ? r.map((x, i) => {
461:       const href = x.url
462:         ? esc(x.url)
463:         : `#/story/${encodeURIComponent(x.storyId)}`;
464: 
465:       const external = x.url
466:         ? 'target="_blank" rel="noopener noreferrer"'
467:         : "";
468: 
469:       return `
470:         <a class="card re

In [3]:
from pathlib import Path
from datetime import datetime

app_path = Path(r"D:\Projects\NewsGPT\frontend\app.js")
source = app_path.read_text(encoding="utf-8")

# Find the existing function and the next section boundary.
start_marker = "  async function viewSearch(q) {"
end_marker = "  // ============ STORY / TIMELINE"

start = source.find(start_marker)

if start == -1:
    raise ValueError("Could not find viewSearch(q). No changes made.")

end = source.find(end_marker, start)

if end == -1:
    raise ValueError(
        "Could not find the story/timeline section boundary. No changes made."
    )

# Create a backup before editing.
backup_path = app_path.with_name(
    f"app.js.backup_{datetime.now():%Y%m%d_%H%M%S}"
)
backup_path.write_text(source, encoding="utf-8")

new_function = r'''  async function viewSearch(q) {
    app.innerHTML = `<div class="container page-pad">
      <p class="crumbs"><a href="#/">← Back to home</a></p>
      <div class="section-head">
        <h2>Search</h2>
        <span class="kicker" id="srCount">Searching…</span>
      </div>
      <div style="max-width:560px;margin-bottom:26px">${searchMarkup(q)}</div>
      <div id="srOut" aria-live="polite"></div>

      <section style="margin-top:42px" aria-labelledby="recommendHeading">
        <div class="section-head">
          <h2 id="recommendHeading">Recommended for You</h2>
          <span class="kicker" id="recCount">Finding recommendations…</span>
        </div>
        <div id="recOut" aria-live="polite"></div>
      </section>
    </div>`;

    initSearch($(".search", app));

    const out = $("#srOut");
    const recOut = $("#recOut");
    const recCount = $("#recCount");

    function renderArticleCards(articles) {
      if (!articles.length) {
        return `<p class="empty">No matching articles found. Try another search.</p>`;
      }

      return articles.map((x, i) => {
        const href = x.url
          ? esc(x.url)
          : x.storyId
            ? `#/story/${encodeURIComponent(x.storyId)}`
            : "#/";

        const external = x.url
          ? 'target="_blank" rel="noopener noreferrer"'
          : "";

        return `
          <a class="card reveal"
             style="--i:${i % 6};margin-bottom:18px"
             href="${href}" ${external}>
            <span class="cat-label">${esc(x.category || "News")} · Article</span>
            <h3>${esc(x.title || "Untitled article")}</h3>
            <p>${esc(x.description || "")}</p>
            <span class="meta">
              <b>${esc(x.publisher || "Unknown publisher")}</b>
              · ${fmtDate(x.publishedAt)}
            </span>
          </a>
        `;
      }).join("");
    }

    // Load search results and recommendations independently.
    const searchTask = (async () => {
      try {
        const results = await searchNews(q);
        $("#srCount").textContent =
          `${results.length} result${results.length === 1 ? "" : "s"} for “${q}”`;
        out.innerHTML = renderArticleCards(results);
      } catch (error) {
        $("#srCount").textContent = "Error";
        out.innerHTML =
          `<p class="empty" role="alert">Search is unavailable right now. Please try again.</p>`;
      }
    })();

    const recommendationTask = (async () => {
      try {
        if (!CFG.apiBase) {
          recCount.textContent = "Unavailable";
          recOut.innerHTML =
            `<p class="empty">Recommendations require the NewsGPT backend.</p>`;
          return;
        }

        const response = await fetch(
          `${CFG.apiBase}/recommend?q=${encodeURIComponent(q)}&top_n=10`
        );

        if (!response.ok) {
          throw new Error(`Recommendation request failed: ${response.status}`);
        }

        const data = await response.json();
        const recommendations = Array.isArray(data)
          ? data
          : (data.results || []);

        recCount.textContent =
          `${recommendations.length} recommendation${recommendations.length === 1 ? "" : "s"}`;

        recOut.innerHTML = renderArticleCards(recommendations);
      } catch (error) {
        console.error("NewsGPT recommendations error:", error);
        recCount.textContent = "Unavailable";
        recOut.innerHTML =
          `<p class="empty">Recommendations couldn't load. Your search results are still available.</p>`;
      }
    })();

    await Promise.all([searchTask, recommendationTask]);

    observeReveals(app);
    window.scrollTo(0, 0);
  }

'''

updated = source[:start] + new_function + source[end:]
app_path.write_text(updated, encoding="utf-8")

print("Updated:", app_path)
print("Backup created:", backup_path)
print("Only viewSearch(q) was replaced.")

ValueError: Could not find the story/timeline section boundary. No changes made.

In [4]:
from pathlib import Path

app_path = Path(r"D:\Projects\NewsGPT\frontend\app.js")
lines = app_path.read_text(encoding="utf-8").splitlines()

start = next(
    i for i, line in enumerate(lines)
    if "async function viewSearch(q)" in line
)

print("viewSearch starts at line:", start + 1)
print("\nFunctions and section markers following viewSearch:\n")

for i in range(start + 1, min(start + 130, len(lines))):
    line = lines[i].strip()

    if (
        line.startswith("function ")
        or line.startswith("async function ")
        or line.startswith("//")
        or line.startswith("/*")
    ):
        print(f"{i + 1}: {line}")

viewSearch starts at line: 448

Functions and section markers following viewSearch:

489: /* ---------- story / timeline ---------- */
490: function viewStory(id) {
527: // save toggle
538: function renderTL() {
564: /* ---------- saved / about / not found ---------- */
565: function viewSaved() {
574: function viewAbout() {


In [5]:
from pathlib import Path
from datetime import datetime

app_path = Path(r"D:\Projects\NewsGPT\frontend\app.js")
source = app_path.read_text(encoding="utf-8")

start_marker = "  async function viewSearch(q) {"
end_marker = "/* ---------- story / timeline ---------- */"

start = source.find(start_marker)
end = source.find(end_marker, start)

if start == -1 or end == -1:
    raise ValueError("Could not safely identify the function boundaries. No changes made.")

# Verify the expected section before editing.
old_section = source[start:end]

if "async function viewSearch(q)" not in old_section:
    raise ValueError("Unexpected source content. No changes made.")

# Create backup.
backup_path = app_path.with_name(
    f"app.js.backup_{datetime.now():%Y%m%d_%H%M%S}"
)
backup_path.write_text(source, encoding="utf-8")

new_function = r'''  async function viewSearch(q) {
    app.innerHTML = `<div class="container page-pad">
      <p class="crumbs"><a href="#/">← Back to home</a></p>
      <div class="section-head">
        <h2>Search</h2>
        <span class="kicker" id="srCount">Searching…</span>
      </div>
      <div style="max-width:560px;margin-bottom:26px">${searchMarkup(q)}</div>
      <div id="srOut" aria-live="polite"></div>

      <section style="margin-top:42px" aria-labelledby="recommendHeading">
        <div class="section-head">
          <h2 id="recommendHeading">Recommended for You</h2>
          <span class="kicker" id="recCount">Loading…</span>
        </div>
        <div id="recOut" aria-live="polite"></div>
      </section>
    </div>`;

    initSearch($(".search", app));

    const out = $("#srOut");
    const recOut = $("#recOut");
    const recCount = $("#recCount");

    function renderArticleCards(articles) {
      if (!articles.length) {
        return `<p class="empty">No matching articles found. Try another search.</p>`;
      }

      return articles.map((x, i) => {
        const href = x.url
          ? esc(x.url)
          : x.storyId
            ? `#/story/${encodeURIComponent(x.storyId)}`
            : "#/";

        const external = x.url
          ? 'target="_blank" rel="noopener noreferrer"'
          : "";

        return `
          <a class="card reveal"
             style="--i:${i % 6};margin-bottom:18px"
             href="${href}" ${external}>
            <span class="cat-label">${esc(x.category || "News")} · Article</span>
            <h3>${esc(x.title || "Untitled article")}</h3>
            <p>${esc(x.description || "")}</p>
            <span class="meta">
              <b>${esc(x.publisher || "Unknown publisher")}</b>
              · ${fmtDate(x.publishedAt)}
            </span>
          </a>
        `;
      }).join("");
    }

    // Search results
    const searchTask = (async () => {
      try {
        const results = await searchNews(q);

        $("#srCount").textContent =
          `${results.length} result${results.length === 1 ? "" : "s"} for “${q}”`;

        out.innerHTML = renderArticleCards(results);
      } catch (error) {
        console.error("NewsGPT search error:", error);
        $("#srCount").textContent = "Error";
        out.innerHTML =
          `<p class="empty" role="alert">Search is unavailable right now. Please try again.</p>`;
      }
    })();

    // Recommendations
    const recommendationTask = (async () => {
      try {
        if (!CFG.apiBase) {
          throw new Error("NewsGPT API is not configured.");
        }

        const response = await fetch(
          `${CFG.apiBase}/recommend?q=${encodeURIComponent(q)}&top_n=10`
        );

        if (!response.ok) {
          throw new Error(`Recommendation request failed: ${response.status}`);
        }

        const data = await response.json();
        const recommendations = Array.isArray(data)
          ? data
          : (data.results || []);

        recCount.textContent =
          `${recommendations.length} recommendation${recommendations.length === 1 ? "" : "s"}`;

        recOut.innerHTML = renderArticleCards(recommendations);
      } catch (error) {
        console.error("NewsGPT recommendation error:", error);
        recCount.textContent = "Unavailable";
        recOut.innerHTML =
          `<p class="empty">Recommendations couldn't load. Search results are still available.</p>`;
      }
    })();

    await Promise.all([searchTask, recommendationTask]);

    observeReveals(app);
    window.scrollTo(0, 0);
  }

'''

updated = source[:start] + new_function + source[end:]

# Basic safety checks before writing.
assert "async function viewSearch(q)" in updated
assert "function viewStory(id)" in updated
assert "function renderTL()" in updated
assert updated.count("async function viewSearch(q)") == 1

app_path.write_text(updated, encoding="utf-8")

print("SUCCESS: viewSearch() updated.")
print("Backup:", backup_path)
print("Story/timeline functions preserved.")

SUCCESS: viewSearch() updated.
Backup: D:\Projects\NewsGPT\frontend\app.js.backup_20261022_065552
Story/timeline functions preserved.


In [6]:
from pathlib import Path

project = Path(r"D:\Projects\NewsGPT")
app_path = project / "frontend" / "app.js"
backend_path = project / "backend" / "app" / "main.py"

js_lines = app_path.read_text(encoding="utf-8").splitlines()
backend_lines = backend_path.read_text(encoding="utf-8").splitlines()

print("=" * 25, "FRONTEND ROUTING", "=" * 25)

for start, end in [(220, 285), (385, 450), (480, 535)]:
    print(f"\n--- app.js lines {start}-{end} ---")
    print("\n".join(
        f"{i + 1}: {js_lines[i]}"
        for i in range(start - 1, min(end, len(js_lines)))
    ))

print("\n" + "=" * 25, "BACKEND ENDPOINTS", "=" * 25)

for i, line in enumerate(backend_lines, start=1):
    if line.startswith("@app.") or line.startswith("def ") or line.startswith("async def "):
        print(f"{i}: {line.strip()}")

========================= FRONTEND ROUTING =========================

--- app.js lines 220-285 ---
220:       let score = 0;
221:       terms.forEach((t) => {
222:         if (hay(s.headline).includes(t)) score += 4;
223:         if (hay(s.category).includes(t)) score += 3;
224:         if (hay(s.dek).includes(t)) score += 1;
225:       });
226:       if (score) out.push({ kind: "story", storyId: s.id, title: s.headline, category: s.category, publisher: "Story timeline · " + s.timeline.length + " articles", publishedAt: lastItem(s).publishedAt, score: score + 2 });
227:       s.timeline.forEach((a) => {
228:         let sc = 0;
229:         terms.forEach((t) => { if (hay(a.headline).includes(t)) sc += 3; if (hay(a.summary).includes(t)) sc += 1; if (hay(a.publisher).includes(t)) sc += 1; });
230:         if (sc) out.push({ kind: "article", storyId: s.id, title: a.headline, category: s.category, publisher: a.publisher, publishedAt: a.publishedAt, score: sc });
231:       });
232:     });

In [7]:
from pathlib import Path

app_path = Path(r"D:\Projects\NewsGPT\frontend\app.js")
lines = app_path.read_text(encoding="utf-8").splitlines()

print("=== ROUTING AND ARTICLE NAVIGATION ===\n")

keywords = [
    "function route",
    "function router",
    "function renderApp",
    "addEventListener(\"hashchange\"",
    "addEventListener('hashchange'",
    "location.hash.slice",
    "location.hash.substring",
    "viewStory(",
    "#/search/",
]

shown = set()

for i, line in enumerate(lines):
    if any(keyword in line for keyword in keywords):
        start = max(0, i - 3)
        end = min(len(lines), i + 9)

        for j in range(start, end):
            if j not in shown:
                print(f"{j + 1}: {lines[j]}")
                shown.add(j)

        print()

=== ROUTING AND ARTICLE NAVIGATION ===

261: }
262:     function submit() {
263:       const q = input.value.trim(); if (!q) return;
264:       close(); location.hash = `#/search/${encodeURIComponent(q)}`;
265:     }
266:     input.addEventListener("input", () => {
267:       clearBtn.hidden = !input.value;
268:       clearTimeout(timer);
269:       const q = input.value.trim();
270:       if (q.length < 2) { close(); return; }
271:       const my = ++seq;
272:       timer = setTimeout(async () => {

558:   }
559: 
560: /* ---------- story / timeline ---------- */
561:   function viewStory(id) {
562:     const s = byId(id); if (!s) return viewNotFound();
563:     const tl = s.timeline, first = tl[0], last = lastItem(s);
564:     const pubs = [...new Set(tl.map((a) => a.publisher))].sort();
565:     app.innerHTML = `<div class="container">
566:       <header class="story-head">
567:         <p class="crumbs"><a href="#/">← Back to home</a> &nbsp;·&nbsp; <a href="#/topic/${slug(s.categor

In [8]:
from pathlib import Path

backend_path = Path(r"D:\Projects\NewsGPT\backend\app\main.py")
backend_lines = backend_path.read_text(encoding="utf-8").splitlines()

print("=== BACKEND DATA AND ENDPOINT STRUCTURE ===\n")

for i, line in enumerate(backend_lines):
    if any(term in line for term in [
        "DATASET_PATH",
        "def get_article_results",
        "def search_news",
        "def recommend_news",
        'df.iloc[',
        'article_id',
    ]):
        print(f"{i + 1}: {line.strip()}")

=== BACKEND DATA AND ENDPOINT STRUCTURE ===

18: DATASET_PATH = RESULT_DIR / "news_dataset.csv"
58: DATASET_PATH,
75: df = pd.read_csv(DATASET_PATH)
99: usecols=["article_id", "cluster"],
103: subset=["article_id", "cluster"]
108: clustered_df["article_id"].astype(str),
122: def get_article_results(scores, top_n):
136: article = df.iloc[index]
148: "articleId": str(article.get("article_id", "")),
203: def search_news(
237: def recommend_news(
272: best_article_id = str(
273: df.iloc[best_index].get("article_id", "")
276: matching_cluster = cluster_map.get(best_article_id)
280: article_id
281: for article_id, cluster_id in cluster_map.items()
286: df["article_id"].astype(str)


In [9]:
from pathlib import Path
from datetime import datetime

backend_path = Path(r"D:\Projects\NewsGPT\backend\app\main.py")
backend = backend_path.read_text(encoding="utf-8")

marker = '@app.get("/categories")'

if '@app.get("/articles/{article_id}")' in backend:
    print("Article-detail endpoint already exists. No changes made.")
elif marker not in backend:
    raise ValueError("Could not find the categories endpoint. No changes made.")
else:
    backup = backend_path.with_name(
        f"main.py.backup_{datetime.now():%Y%m%d_%H%M%S}"
    )
    backup.write_text(backend, encoding="utf-8")

    endpoint = '''

@app.get("/articles/{article_id}")
def get_article(article_id: str):
    """Retrieve one article using its article ID."""

    matches = df[df["article_id"].astype(str) == article_id]

    if matches.empty:
        raise HTTPException(
            status_code=404,
            detail="Article not found."
        )

    article = matches.iloc[0]

    def safe_value(column, default=""):
        value = article.get(column, default)
        return default if pd.isna(value) else str(value)

    return {
        "articleId": safe_value("article_id"),
        "title": safe_value("title", "Untitled article"),
        "description": safe_value("description"),
        "content": safe_value("content"),
        "category": safe_value("category", "News"),
        "publisher": safe_value("source_name", "Unknown publisher"),
        "publishedAt": safe_value("published_at"),
        "url": safe_value("url"),
        "imageUrl": safe_value("url_to_image"),
    }

'''

    backend_path.write_text(
        backend.replace(marker, endpoint + marker, 1),
        encoding="utf-8"
    )

    print("Article-detail endpoint added.")
    print("Backup:", backup)

Article-detail endpoint added.
Backup: D:\Projects\NewsGPT\backend\app\main.py.backup_20261022_070354


In [10]:
from pathlib import Path
from datetime import datetime

app_path = Path(r"D:\Projects\NewsGPT\frontend\app.js")
source = app_path.read_text(encoding="utf-8")

story_marker = "/* ---------- story / timeline ---------- */"
route_marker = '      case "story": viewStory(decodeURIComponent(parts[1] || "")); navKey = ""; break;'

if "async function viewArticle(articleId)" in source:
    print("Article-detail function already exists. No changes made.")
elif story_marker not in source or route_marker not in source:
    raise ValueError(
        "Could not find the expected frontend markers. No changes made."
    )
else:
    backup_path = app_path.with_name(
        f"app.js.backup_{datetime.now():%Y%m%d_%H%M%S}"
    )
    backup_path.write_text(source, encoding="utf-8")

    article_function = r'''
  /* ---------- real article detail ---------- */
  async function viewArticle(articleId) {
    app.innerHTML = `<div class="container page-pad">
      <p class="crumbs"><a href="#/">← Back to home</a></p>
      <div id="articleDetail" aria-live="polite">
        <p class="empty">Loading article…</p>
      </div>
    </div>`;

    window.scrollTo(0, 0);

    try {
      const response = await fetch(
        `${CFG.apiBase}/articles/${encodeURIComponent(articleId)}`
      );

      if (!response.ok) {
        if (response.status === 404) {
          throw new Error("This article could not be found.");
        }
        throw new Error("Unable to load this article.");
      }

      const article = await response.json();
      const detail = $("#articleDetail");

      const originalLink = article.url
        ? `<a class="btn btn--primary"
              href="${esc(article.url)}"
              target="_blank"
              rel="noopener noreferrer">
              Read Original Article ↗
           </a>`
        : `<p class="empty">The original article link is unavailable.</p>`;

      const image = article.imageUrl
        ? `<img src="${esc(article.imageUrl)}"
                alt=""
                loading="lazy"
                style="display:block;width:100%;max-height:420px;object-fit:cover;border-radius:12px;margin:24px 0"
                onerror="this.style.display='none'">`
        : "";

      detail.innerHTML = `
        <article class="article-detail">
          <span class="cat-label">${esc(article.category || "News")}</span>

          <h1 style="margin:16px 0">${esc(article.title || "Untitled article")}</h1>

          <div class="meta" style="margin-bottom:20px">
            <b>${esc(article.publisher || "Unknown publisher")}</b>
            ${article.publishedAt
              ? ` · ${fmtDate(article.publishedAt)}`
              : ""}
          </div>

          ${image}

          <section aria-labelledby="article-summary-heading"
                   style="max-width:800px;margin:28px 0">
            <h2 id="article-summary-heading">Summary</h2>
            <p style="font-size:1.08rem;line-height:1.8">
              ${esc(article.description || "No summary is available in the dataset for this article.")}
            </p>
          </section>

          ${originalLink}

          <section style="margin-top:48px"
                   aria-labelledby="article-recommendations-heading">
            <div class="section-head">
              <h2 id="article-recommendations-heading">Related Recommendations</h2>
              <span class="kicker" id="articleRecCount">Loading…</span>
            </div>
            <div id="articleRecOut" aria-live="polite"></div>
          </section>
        </article>
      `;

      loadArticleRecommendations(article);

      observeReveals(app);
    } catch (error) {
      console.error("Article detail error:", error);

      $("#articleDetail").innerHTML = `
        <p class="empty" role="alert">${esc(error.message || "Unable to load this article.")}</p>
        <p><a href="#/">Return to NewsGPT</a></p>
      `;
    }
  }

  async function loadArticleRecommendations(article) {
    const output = $("#articleRecOut");
    const count = $("#articleRecCount");

    try {
      const response = await fetch(
        `${CFG.apiBase}/recommend?q=${encodeURIComponent(article.title)}&top_n=10`
      );

      if (!response.ok) {
        throw new Error("Recommendation request failed.");
      }

      const data = await response.json();
      const recommendations = (Array.isArray(data) ? data : data.results || [])
        .filter(item => String(item.articleId) !== String(article.articleId));

      count.textContent =
        `${recommendations.length} related article${recommendations.length === 1 ? "" : "s"}`;

      if (!recommendations.length) {
        output.innerHTML = `<p class="empty">No related articles found.</p>`;
        return;
      }

      output.innerHTML = recommendations.map((item, index) => `
        <a class="card reveal"
           style="--i:${index % 6};margin-bottom:18px"
           href="#/article/${encodeURIComponent(item.articleId)}">
          <span class="cat-label">${esc(item.category || "News")} · Recommended</span>
          <h3>${esc(item.title || "Untitled article")}</h3>
          <p>${esc(item.description || "")}</p>
          <span class="meta">
            <b>${esc(item.publisher || "Unknown publisher")}</b>
            ${item.publishedAt ? ` · ${fmtDate(item.publishedAt)}` : ""}
          </span>
        </a>
      `).join("");

      observeReveals(output);
    } catch (error) {
      console.error("Article recommendations error:", error);
      count.textContent = "Unavailable";
      output.innerHTML =
        `<p class="empty">Recommendations couldn't load. You can still read the article above.</p>`;
    }
  }

'''

    updated = source.replace(
        story_marker,
        article_function + "\n  " + story_marker,
        1
    )

    updated = updated.replace(
        route_marker,
        '      case "article": viewArticle(decodeURIComponent(parts[1] || "")); navKey = ""; break;\n'
        + route_marker,
        1
    )

    assert 'case "article": viewArticle(' in updated
    assert "function viewStory(id)" in updated

    app_path.write_text(updated, encoding="utf-8")

    print("Article-detail page and route added.")
    print("Backup:", backup_path)

Article-detail page and route added.
Backup: D:\Projects\NewsGPT\frontend\app.js.backup_20261022_070419


In [11]:
from pathlib import Path
from datetime import datetime

app_path = Path(r"D:\Projects\NewsGPT\frontend\app.js")
source = app_path.read_text(encoding="utf-8")

old = '''        const href = x.url
          ? esc(x.url)
          : x.storyId
            ? `#/story/${encodeURIComponent(x.storyId)}`
            : "#/";

        const external = x.url
          ? 'target="_blank" rel="noopener noreferrer"'
          : "";'''

new = '''        const href = x.articleId
          ? `#/article/${encodeURIComponent(x.articleId)}`
          : x.storyId
            ? `#/story/${encodeURIComponent(x.storyId)}`
            : x.url
              ? esc(x.url)
              : "#/";

        const external = x.articleId || x.storyId
          ? ""
          : x.url
            ? 'target="_blank" rel="noopener noreferrer"'
            : "";'''

if old not in source:
    print("Could not find the expected search-card link block.")
    print("No changes made. Share the current renderArticleCards() function.")
else:
    backup_path = app_path.with_name(
        f"app.js.backup_{datetime.now():%Y%m%d_%H%M%S}"
    )
    backup_path.write_text(source, encoding="utf-8")

    app_path.write_text(
        source.replace(old, new, 1),
        encoding="utf-8"
    )

    print("Search cards now open NewsGPT article details.")
    print("Backup:", backup_path)

Search cards now open NewsGPT article details.
Backup: D:\Projects\NewsGPT\frontend\app.js.backup_20261022_070440


In [12]:
from pathlib import Path
from datetime import datetime

app_path = Path(r"D:\Projects\NewsGPT\frontend\app.js")
source = app_path.read_text(encoding="utf-8")

old = '''  function go(r) {
  close();

  // Real backend articles have their own source URLs.
  if (r.url) {
    window.open(r.url, "_blank", "noopener,noreferrer");
    return;
  }

  // Preserve navigation for the existing sample stories.
  if (r.storyId) {
    location.hash = `#/story/${r.storyId}`;
  }
}'''

new = '''    function go(r) {
      close();

      // Real news articles open inside NewsGPT first.
      if (r.articleId) {
        location.hash = `#/article/${encodeURIComponent(r.articleId)}`;
        return;
      }

      // Preserve existing sample-story navigation.
      if (r.storyId) {
        location.hash = `#/story/${encodeURIComponent(r.storyId)}`;
        return;
      }

      // Fallback for any external article without an ID.
      if (r.url) {
        window.open(r.url, "_blank", "noopener,noreferrer");
      }
    }'''

if old not in source:
    print("Could not find the expected autocomplete function.")
    print("No changes made. Share the current go(r) function.")
else:
    backup_path = app_path.with_name(
        f"app.js.backup_{datetime.now():%Y%m%d_%H%M%S}"
    )
    backup_path.write_text(source, encoding="utf-8")

    app_path.write_text(
        source.replace(old, new, 1),
        encoding="utf-8"
    )

    print("Autocomplete now opens NewsGPT article details.")
    print("Backup:", backup_path)

Autocomplete now opens NewsGPT article details.
Backup: D:\Projects\NewsGPT\frontend\app.js.backup_20261022_070456


In [13]:
from pathlib import Path

app_path = Path(r"D:\Projects\NewsGPT\frontend\app.js")
lines = app_path.read_text(encoding="utf-8").splitlines()

start = next(
    i for i, line in enumerate(lines)
    if "async function viewSearch(q)" in line
)

end = next(
    i for i in range(start + 1, len(lines))
    if "story / timeline" in lines[i]
)

print("\n".join(
    f"{i + 1}: {lines[i]}"
    for i in range(start, end)
))

454:   async function viewSearch(q) {
455:     app.innerHTML = `<div class="container page-pad">
456:       <p class="crumbs"><a href="#/">← Back to home</a></p>
457:       <div class="section-head">
458:         <h2>Search</h2>
459:         <span class="kicker" id="srCount">Searching…</span>
460:       </div>
461:       <div style="max-width:560px;margin-bottom:26px">${searchMarkup(q)}</div>
462:       <div id="srOut" aria-live="polite"></div>
463: 
464:       <section style="margin-top:42px" aria-labelledby="recommendHeading">
465:         <div class="section-head">
466:           <h2 id="recommendHeading">Recommended for You</h2>
467:           <span class="kicker" id="recCount">Loading…</span>
468:         </div>
469:         <div id="recOut" aria-live="polite"></div>
470:       </section>
471:     </div>`;
472: 
473:     initSearch($(".search", app));
474: 
475:     const out = $("#srOut");
476:     const recOut = $("#recOut");
477:     const recCount = $("#recCount");
478: 
479: 

In [14]:
from pathlib import Path
import shutil

app_js = Path(r"D:\Projects\NewsGPT\frontend\app.js")
source = app_js.read_text(encoding="utf-8")

start_marker = "  async function viewSearch(q) {"
end_marker = "  /* ---------- real article detail ---------- */"

start = source.find(start_marker)
end = source.find(end_marker, start)

if start == -1 or end == -1:
    print("Could not safely locate viewSearch(). No changes made.")
else:
    search_function = source[start:end]

    if "const recommendationTask" not in search_function:
        print("The search-page recommendation task is already absent.")
    else:
        # Remove the recommendation section from the search-page HTML.
        html_start = search_function.find(
            '      <section style="margin-top:42px" aria-labelledby="recommendHeading">'
        )
        html_end = search_function.find(
            "    initSearch($('.search', app));"
        )

        if html_start == -1:
            html_start = search_function.find(
                '      <section style="margin-top:42px" aria-labelledby="recommendHeading">'
            )

        if html_start == -1 or html_end == -1:
            print("Could not safely locate the recommendation markup. No changes made.")
        else:
            updated = search_function[:html_start] + search_function[html_end:]

            # Remove recommendation-only DOM references.
            updated = updated.replace(
                '    const recOut = $("#recOut");\n'
                '    const recCount = $("#recCount");\n',
                ""
            )

            # Remove the recommendation-fetching task.
            task_start = updated.find("    // Recommendations\n")
            task_end = updated.find(
                "    await Promise.all([searchTask, recommendationTask]);"
            )

            if task_start == -1 or task_end == -1:
                print("Could not safely locate the recommendation task. No changes made.")
            else:
                updated = (
                    updated[:task_start]
                    + "    await searchTask;\n"
                    + updated[task_end + len(
                        "    await Promise.all([searchTask, recommendationTask]);"
                    ):]
                )

                backup = app_js.with_name("app.js.before_search_recommendation_change.bak")
                shutil.copy2(app_js, backup)

                app_js.write_text(
                    source[:start] + updated + source[end:],
                    encoding="utf-8"
                )

                print("Updated viewSearch(): recommendations are no longer shown on search results.")
                print(f"Backup created: {backup}")

Could not safely locate the recommendation markup. No changes made.


In [15]:
from pathlib import Path

app_js = Path(r"D:\Projects\NewsGPT\frontend\app.js")
source = app_js.read_text(encoding="utf-8")

start = source.find("  async function viewSearch(q) {")
end = source.find("/* ---------- real article detail ---------- */", start)

if start == -1:
    print("ERROR: Could not find viewSearch(q).")
elif end == -1:
    print("ERROR: Could not find the article detail section.")
else:
    print(source[start:end])

  async function viewSearch(q) {
    app.innerHTML = `<div class="container page-pad">
      <p class="crumbs"><a href="#/">← Back to home</a></p>
      <div class="section-head">
        <h2>Search</h2>
        <span class="kicker" id="srCount">Searching…</span>
      </div>
      <div style="max-width:560px;margin-bottom:26px">${searchMarkup(q)}</div>
      <div id="srOut" aria-live="polite"></div>

      <section style="margin-top:42px" aria-labelledby="recommendHeading">
        <div class="section-head">
          <h2 id="recommendHeading">Recommended for You</h2>
          <span class="kicker" id="recCount">Loading…</span>
        </div>
        <div id="recOut" aria-live="polite"></div>
      </section>
    </div>`;

    initSearch($(".search", app));

    const out = $("#srOut");
    const recOut = $("#recOut");
    const recCount = $("#recCount");

    function renderArticleCards(articles) {
      if (!articles.length) {
        return `<p class="empty">No matching articles fo

In [16]:
from pathlib import Path
import shutil

app_js = Path(r"D:\Projects\NewsGPT\frontend\app.js")
source = app_js.read_text(encoding="utf-8")

start = source.find("  async function viewSearch(q) {")
end = source.find("/* ---------- real article detail ---------- */", start)

if start == -1 or end == -1:
    print("STOP: Could not safely locate the search function.")
else:
    block = source[start:end]

    # Remove the recommendation section from the page HTML.
    html_start = block.find(
        '      <section style="margin-top:42px" aria-labelledby="recommendHeading">'
    )
    html_end = block.find(
        '    initSearch($(".search", app));'
    )

    # Confirm all expected code exists before editing.
    required = [
        "    const recOut = $(\"#recOut\");",
        '    const recCount = $("#recCount");',
        "    // Recommendations",
        "    const recommendationTask = (async () => {",
        "    await Promise.all([searchTask, recommendationTask]);",
    ]

    missing = [item for item in required if item not in block]

    if html_start == -1 or html_end == -1 or missing:
        print("STOP: The code differs from the expected version.")
        print("Missing elements:", missing)
        print("No changes made.")
    else:
        # Remove recommendation HTML.
        block = block[:html_start] + block[html_end:]

        # Remove recommendation DOM references.
        block = block.replace(
            '    const recOut = $("#recOut");\n'
            '    const recCount = $("#recCount");\n',
            ""
        )

        # Remove the recommendation-fetching task.
        task_start = block.find("    // Recommendations")
        task_end = block.find(
            "    await Promise.all([searchTask, recommendationTask]);"
        )

        block = (
            block[:task_start]
            + "    await searchTask;"
            + block[task_end + len(
                "    await Promise.all([searchTask, recommendationTask]);"
            ):]
        )

        # Save a backup and write the updated function.
        backup = app_js.with_name("app.js.before_search_recommendation_change.bak")
        shutil.copy2(app_js, backup)

        app_js.write_text(
            source[:start] + block + source[end:],
            encoding="utf-8"
        )

        print("SUCCESS: Removed recommendations from the search page.")
        print("Search results and article navigation were preserved.")
        print("Backup:", backup)

SUCCESS: Removed recommendations from the search page.
Search results and article navigation were preserved.
Backup: D:\Projects\NewsGPT\frontend\app.js.before_search_recommendation_change.bak


In [17]:
from pathlib import Path
import shutil

frontend = Path(r"D:\Projects\NewsGPT\frontend")
app_js = frontend / "app.js"
backup = frontend / "app.js.before_search_recommendation_change.bak"

if backup.exists():
    shutil.copy2(backup, app_js)
    print("SUCCESS: Previous app.js restored.")
    print("Refresh the website and check whether your UI has returned.")
else:
    print("Backup not found. No files were changed.")
    print("We'll recover the code using another method.")

SUCCESS: Previous app.js restored.
Refresh the website and check whether your UI has returned.
